In [ ]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Ambil konfigurasi database dari folder utama project kalian
sys.path.append(os.path.abspath('..'))
from config import get_db_config

config = get_db_config()

# 1. Koneksi ke Database Baru (Fase Migrasi Sekarang)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)

# 2. Koneksi ke Database Masa Depan (DB_FUTURE)
# Catatan: Pastikan di file config.py kalian sudah ada key 'db_future' ya!
db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)

print(f"✅ Sukses Terhubung ke Database Baru : {config['db_new']['database']}")
print(f"🚀 Sukses Terhubung ke DB_FUTURE     : {config['db_future']['database']}")

: 

In [ ]:
tables_to_check = [
    # --- Bagian Cimut ---
    "karyawan", 
    "keluarga_karyawan", 
    "bidang_kategori", 
    "bidang_link",
    
    # --- Bagian Afrida ---
    "periode", 
    "parameter_nilai", 
    "kabupaten", 
    "kecamatan",
    
    # --- Bagian Hanif ---
    "division_user", 
    "model_has_roles", 
    "model_has_permissions", 
    "kelurahan"
]

In [ ]:
# === Cell 2: Inspeksi Detektor Pintar dengan Prioritas Target Revisi di Atas ===
import pandas as pd
import numpy as np

print("================================================================================")
print(" 🕵️‍♂️ FORENSIK ARSITEKTUR KONTROL: DETEKTOR AUTO-SMART COMPILER (REVISI PRIORITY) 🕵️‍♂️ ")
print("================================================================================")

# List penampung data di memori untuk keperluan sorting visualisasi
revisi_tables_queue = []
identical_tables_queue = []

# --- TAHAP A: PROSES PEN ARIKAN DATA & EVALUASI STRUKTUR DI BELAKANG LAYAR ---
for table in tables_to_check:
    try:
        # 1. Ambil data asli dari DB_NEW untuk kebutuhan .info() dan sampel isi data
        query = f"SELECT * FROM `{table}`"
        df_real_data = pd.read_sql(query, db_new)
        
        # 2. Tarik Struktur Fisik Kolom dari DB_NEW
        query_new_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_new']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_new = pd.read_sql(query_new_struct, db_new).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_NEW
        pk_referenced_list = []
        for idx, row_skri in df_struct_new.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_new']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar = pd.read_sql(lookup_fk_query, db_new)
                if not df_relasi_luar.empty:
                    pk_referenced_list.append("\n".join(df_relasi_luar['relasi'].tolist()))
                else:
                    pk_referenced_list.append("-")
            else:
                pk_referenced_list.append("-")
        df_struct_new['Tabel Yang nge-FK (DB_NEW)'] = pk_referenced_list

        # 3. Tarik Struktur Fisik Kolom dari DB_FUTURE
        query_future_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_future']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct_future = pd.read_sql(query_future_struct, db_future).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Lacak tabel anak yang nge-FK ke PK tabel ini di DB_FUTURE
        pk_referenced_list_future = []
        for idx, row_skri in df_struct_future.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query_future = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_future']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar_future = pd.read_sql(lookup_fk_query_future, db_future)
                if not df_relasi_luar_future.empty:
                    pk_referenced_list_future.append("\n".join(df_relasi_luar_future['relasi'].tolist()))
                else:
                    pk_referenced_list_future.append("-")
            else:
                pk_referenced_list_future.append("-")
        df_struct_future['Tabel Yang nge-FK (DB_FUTURE)'] = pk_referenced_list_future

        # 4. Deep Comparison Kesamaan Jeroan Kolom dasar
        cols_to_compare = ['Nama Kolom', 'Tipe Data MySQL', 'Aturan Nullability & Increment', 'Status Kunci', 'Rujukan Induk (FK Origin)', 'Daftar Pilihan ENUM']
        
        is_structure_identical = False
        if not df_struct_new.empty and not df_struct_future.empty:
            is_structure_identical = df_struct_new[cols_to_compare].equals(df_struct_future[cols_to_compare])

        # Wadah paket data tabel untuk di-render nanti
        table_package = {
            'name': table,
            'df_real_data': df_real_data,
            'df_struct_new': df_struct_new,
            'df_struct_future': df_struct_future,
            'is_identical': is_structure_identical
        }

        # 🔥 FILTER SAKTI CIMUT: Pisahkan antrean, utamakan yang bermasalah (revisi) ke atas!
        if is_structure_identical:
            identical_tables_queue.append(table_package)
        else:
            revisi_tables_queue.append(table_package)
            
    except Exception as e:
        print(f"❌ Gagal menganalisis awal tabel `{table}`: {e}")

# --- TAHAP B: MULAI PEN TAMPILAN VISUALISASI BERDASARKAN ANT REAN PRIORITAS ---

# 🚨 1. KELOMPOK UTAMA (PALING ATAS): DAFTAR TABEL YANG WAJIB DIREVISI 🚨
if revisi_tables_queue:
    print("\n" + "!"*80)
    print(f"🚨 [🔥 REVISI PRIORITY BOARD] TERDETEKSI {len(revisi_tables_queue)} TABEL BERBEDA - HARUS SEGERA DIPERBAIKI!")
    print("!"*80)
    
    for pkg in revisi_tables_queue:
        print(f"\n================================================================================")
        print(f"⚠️  [STATUS: TARGET REVISI] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL SAAT INI] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"🔮 3. [⚠️ TARGET BLUEPRINT RUJUKAN] Tinjau Struktur yang Benar pada DB_FUTURE di bawah:")
        if pkg['df_struct_future'].empty:
            print("❌ ERROR: Tabel ini tidak ditemukan / belum dibuat sama sekali di DB_FUTURE!")
        else:
            display(pkg['df_struct_future'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"📸 4. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)

# ✨ 2. KELOMPOK KEDUA (BAW AH): DAFTAR TABEL YANG SUDAH AMAN IDENTIK ✨
if identical_tables_queue:
    print("\n" + "="*80)
    print(f"✨ [🟢 PERFECTLY MATCHED BOARD] SEBANYAK {len(identical_tables_queue)} TABEL STRUKTURNYA SUDAH IDENTIK SAMA PERSIS!")
    print("="*80)
    
    for pkg in identical_tables_queue:
        print(f"\n================================================================================")
        print(f"✅ [STATUS: AMAN IDENTIK] TABEL: {pkg['name'].upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        pkg['df_real_data'].info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. [TABEL REAL & FINAL] Arsitektur Fisik Kolom pada DB_NEW:")
        display(pkg['df_struct_new'].style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print("✨ INDICATOR: [ DB_NEW ] & [ DB_FUTURE ] STRUKTUR SUDAH IDENTIK SAMA PERSIS 100% ✨")
        print("ℹ️  Tabel DB_FUTURE otomatis disembunyikan untuk kenyamanan pemandangan layar monitor.")
        print("\n" + "-"*60)
        
        print(f"📸 3. Sampel Isi Data Real di DB_NEW:")
        display(pkg['df_real_data'] if pkg['df_real_data'].empty else pkg['df_real_data'])
        print("\n" + "="*80)